# Task 3 – Sales Prediction (Simple and Corrected)

**Goal:** Predict Sales using two beginner-friendly machine learning models:
1. Linear Regression
2. Random Forest Regression

**Please run every code cell from top to bottom.** This version avoids complicated `Pipeline`, `ColumnTransformer`, and `OneHotEncoder` steps.


## Step 1 – Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported successfully!")


## Step 2 – Upload Your Dataset

Click **Choose Files** and select your `global_superstore_cleaned` dataset. It can be a CSV or Excel file.


In [ ]:
uploaded = files.upload()

file_name = next(iter(uploaded))

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Please upload a CSV or Excel file.")

print("Dataset loaded successfully!")
print("File name:", file_name)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


## Step 3 – Clean Column Names and Check the Dataset

This makes column names consistent. For example, `Order Date` becomes `order_date`.


In [ ]:
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print("Columns in the dataset:")
print(df.columns.tolist())

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## Step 4 – Prepare the Sales Target and Features

- **Target (`y`)** is the value we want to predict: `sales`.
- **Features (`X`)** are the columns used to predict Sales.
- We use common columns such as quantity, discount, shipping cost, and date parts when available.
- We do not use `profit` or `sales` as input features.


In [ ]:
if "sales" not in df.columns:
    raise ValueError(
        "Could not find a 'sales' column. Please check the column names printed in Step 3."
    )

df["sales"] = pd.to_numeric(df["sales"], errors="coerce")

# Add date features if an order date column is available
if "order_date" in df.columns:
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df["order_year"] = df["order_date"].dt.year
    df["order_month"] = df["order_date"].dt.month
    df["order_quarter"] = df["order_date"].dt.quarter
    df["order_dayofweek"] = df["order_date"].dt.dayofweek

# Use simple numeric predictors that exist in this dataset
possible_features = [
    "quantity",
    "discount",
    "shipping_cost",
    "order_year",
    "order_month",
    "order_quarter",
    "order_dayofweek"
]

features = [col for col in possible_features if col in df.columns]

if not features:
    raise ValueError(
        "No expected feature columns were found. Check the column names from Step 3."
    )

# Make selected features numeric
for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep only target and selected features; remove rows without Sales
model_data = df[features + ["sales"]].dropna(subset=["sales"]).copy()

# Fill missing feature values with each column's median
for col in features:
    model_data[col] = model_data[col].fillna(model_data[col].median())

# Remove any feature column that has no usable values
features = [col for col in features if model_data[col].notna().any()]
model_data = model_data.dropna(subset=features)

X = model_data[features]
y = model_data["sales"]

print("Features used:", features)
print("Target: sales")
print("Rows available for modelling:", len(model_data))

display(model_data.head())


## Step 5 – Split Data into Training and Testing Sets

- 80% of rows are used to train the model.
- 20% of rows are used to test it on data the model did not train on.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## Step 6 – Train the Linear Regression Model


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression Results")
print("RMSE:", round(linear_rmse, 2))
print("MAE :", round(linear_mae, 2))
print("R²  :", round(linear_r2, 4))


## Step 7 – Train the Random Forest Model

Random Forest is a second model. We will compare its results with Linear Regression.


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("RMSE:", round(rf_rmse, 2))
print("MAE :", round(rf_mae, 2))
print("R²  :", round(rf_r2, 4))


## Step 8 – Compare Both Models

- **RMSE:** lower is better.
- **MAE:** lower is better.
- **R²:** higher is generally better.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "RMSE": [linear_rmse, rf_rmse],
    "MAE": [linear_mae, rf_mae],
    "R2": [linear_r2, rf_r2]
})

display(comparison.sort_values("RMSE").reset_index(drop=True))

best_model_name = "Random Forest" if rf_rmse < linear_rmse else "Linear Regression"
best_predictions = rf_pred if rf_rmse < linear_rmse else linear_pred

print("Best model by RMSE:", best_model_name)


## Step 9 – Plot Actual Sales vs Predicted Sales


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_predictions, alpha=0.5)

line_min = min(y_test.min(), best_predictions.min())
line_max = max(y_test.max(), best_predictions.max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle="--")

plt.title("Actual vs Predicted Sales – " + best_model_name)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.tight_layout()
plt.show()


## Step 10 – Feature Importance (Random Forest)


In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(8, 5))
plt.barh(importance["Feature"], importance["Importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance – Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## Step 11 – Business Insights


In [ ]:
print("BUSINESS INSIGHTS")
print("-----------------")
print("1. Best model based on RMSE:", best_model_name)
print("2. Lower RMSE and MAE mean smaller prediction errors.")
print("3. R² helps show how much sales variation the model explains.")
print("4. Most important feature in Random Forest:", importance.iloc[0]["Feature"])
print("5. These predictions are estimates and should support, not replace, business judgement.")


## Step 12 – Conclusion

In this task, we:
- Loaded the Global Superstore dataset
- Checked column names and missing values
- Selected numeric features to predict Sales
- Split data into training and testing sets
- Trained Linear Regression and Random Forest models
- Compared RMSE, MAE and R²
- Created an Actual vs Predicted chart
- Reviewed Random Forest feature importance

**Task 3 is complete when all cells run successfully.**

To save your work in Colab, use **File → Save** or press **Ctrl + S**.
